# Paper10 frontier_random050 50x24/h5 Colab run

This notebook runs the next Paper10 `frontier_random050` 50x24/h5 experiment in Google Colab. It mounts Google Drive, clones or updates the reviewer repository, validates the full Bishan data layout, generates 50-state by 24-candidate value labels with horizon 5, trains the value head, runs rollout gates, and packages the run outputs.

Required full data must be available in Google Drive under one `DATA_ROOT` directory containing `tool2/transitions.npz`, `tool2/pairwise.npz`, `dem_slope_analysis/output/DLTB_with_slope.gpkg` or `dem_slope_analysis/output/DLTB_with_slope.shp`, `results_real/blocks/`, and `townships.json`. Outputs are written to Google Drive, not Colab ephemeral storage.

Expected runtime depends on the Colab GPU and Drive I/O. The long-running label generation and rollout cells may take hours. Intermediate `.partial.npz` and `.partial.json` files are written for monitoring only; the pipeline skips work when final artifacts already exist, but label generation does not resume from a partial file.


In [ ]:
# Parameters

REPO_URL = "https://github.com/zhouning/paper10-geojepa-mpc-farmland-layout.git"
REPO_BRANCH = "main"

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/paper10_frontier_random050_50x24_h5"
DATA_ROOT = "/content/drive/MyDrive/paper10_full_bishan_data"
RUN_NAME = "frontier_random050_50x24_h5_seed45"

n_states=50
candidate_actions=24
label_horizon=5
label_seed=45
training_seed=3045
gamma = 0.99
frontier_fraction = 0.5
candidate_mode = "frontier_random"
progress_every = 1

training_epochs = 3
training_batch_size = 16
transition_samples = 6000
pairwise_states = 50
pairwise_subsample = 24
n_pairs = 8

gate_rollout_steps = 20
final_rollout_steps = 100
rollout_horizon = 5
rollout_top_k = 50
rollout_seed = 0
rollout_seeds_optional='1-4'
run_optional_seeds = False
blend_gate_weights = [0.05, 0.10]

assert n_states == 50
assert candidate_actions == 24
assert label_horizon == 5
assert label_seed == 45
assert training_seed == 3045


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile


CONTENT_DIR = Path("/content")
REPO_DIR = CONTENT_DIR / "paper10-geojepa-mpc-farmland-layout"
DRIVE_PROJECT = Path(DRIVE_PROJECT_DIR)
DATA_ROOT_PATH = Path(DATA_ROOT)
RUN_DIR = DRIVE_PROJECT / "runs" / RUN_NAME
LOG_DIR = RUN_DIR / "logs"
REPORT_DIR = RUN_DIR / "reports"
PACKAGE_DIR = RUN_DIR / "packages"

for path in (DRIVE_PROJECT, RUN_DIR, LOG_DIR, REPORT_DIR, PACKAGE_DIR):
    path.mkdir(parents=True, exist_ok=True)


def run_cmd(args, cwd=None, log_path=None, env=None):
    cwd = Path(cwd or REPO_DIR)
    cmd_text = " ".join(str(part) for part in args)
    print(f"$ {cmd_text}")
    started = time.time()
    log_file = None
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_file = log_path.open("w", encoding="utf-8")
        log_file.write(f"$ {cmd_text}\n")
        log_file.flush()
    process = subprocess.Popen(
        [str(part) for part in args],
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="")
            if log_file is not None:
                log_file.write(line)
                log_file.flush()
    finally:
        if process.stdout is not None:
            process.stdout.close()
        return_code = process.wait()
        elapsed = time.time() - started
        if log_file is not None:
            log_file.write(f"\n[exit={return_code}] elapsed_sec={elapsed:.2f}\n")
            log_file.close()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}: {cmd_text}")
    return return_code


def run_if_missing(output_path, args, cwd=None, log_name=None):
    output_path = Path(output_path)
    if output_path.exists():
        print(f"Skipping because final artifact exists: {output_path}")
        return "skipped"
    log_path = LOG_DIR / log_name if log_name else None
    run_cmd(args, cwd=cwd, log_path=log_path)
    if not output_path.exists():
        raise FileNotFoundError(f"Expected final artifact was not created: {output_path}")
    return "ran"


print(f"Repository path: {REPO_DIR}")
print(f"Data root: {DATA_ROOT_PATH}")
print(f"Run directory: {RUN_DIR}")


## Clone or update repository

The equivalent shell operations are `git clone`, `git fetch origin`, checkout, and fast-forward pull. The code uses `subprocess` so paths remain explicit in Colab.


In [ ]:
if not REPO_DIR.exists():
    run_cmd(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], cwd=CONTENT_DIR, log_path=LOG_DIR / "git_clone.log")
else:
    run_cmd(["git", "fetch", "origin"], cwd=REPO_DIR, log_path=LOG_DIR / "git_fetch.log")
    run_cmd(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR, log_path=LOG_DIR / "git_checkout.log")
    run_cmd(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_DIR, log_path=LOG_DIR / "git_pull.log")

run_cmd(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, log_path=LOG_DIR / "git_head.log")


## Install dependencies

Command preview: `pip install -r requirements.txt`.


In [ ]:
run_cmd([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO_DIR, log_path=LOG_DIR / "pip_install.log")


In [ ]:
# Full-data layout validation.
required_paths = {
    "tool2/transitions.npz": DATA_ROOT_PATH / "tool2" / "transitions.npz",
    "tool2/pairwise.npz": DATA_ROOT_PATH / "tool2" / "pairwise.npz",
    "results_real/blocks": DATA_ROOT_PATH / "results_real" / "blocks",
    "townships.json": DATA_ROOT_PATH / "townships.json",
}

dltb_gpkg = DATA_ROOT_PATH / "dem_slope_analysis" / "output" / "DLTB_with_slope.gpkg"
dltb_shp = DATA_ROOT_PATH / "dem_slope_analysis" / "output" / "DLTB_with_slope.shp"

missing = []
for label, path in required_paths.items():
    if not path.exists():
        missing.append(f"{label}: {path}")
if not (dltb_gpkg.exists() or dltb_shp.exists()):
    missing.append(f"dem_slope_analysis/output/DLTB_with_slope.gpkg or dem_slope_analysis/output/DLTB_with_slope.shp under {DATA_ROOT_PATH}")

if missing:
    raise FileNotFoundError("Missing full-data inputs:\n" + "\n".join(missing))

blocks_dir = required_paths["results_real/blocks"]
block_entries = [item for item in blocks_dir.iterdir() if not item.name.startswith(".")]
if not block_entries:
    raise FileNotFoundError(f"results_real/blocks is empty: {blocks_dir}")

with required_paths["townships.json"].open(encoding="utf-8") as handle:
    townships = json.load(handle)
if not isinstance(townships, dict) or not townships:
    raise ValueError("townships.json must be a non-empty object")

DATA_LAYOUT = {
    "data_root": str(DATA_ROOT_PATH),
    "transition_path": str(required_paths["tool2/transitions.npz"]),
    "tool2_pairwise_path": str(required_paths["tool2/pairwise.npz"]),
    "dltb_path": str(dltb_shp if dltb_shp.exists() else dltb_gpkg),
    "blocks_dir": str(blocks_dir),
    "townships": len(townships),
}
(RUN_DIR / "data_layout.json").write_text(json.dumps(DATA_LAYOUT, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(DATA_LAYOUT, indent=2, sort_keys=True))


## Repository smoke test

Command preview: `pytest paper10_geojepa_mpc/tests -q -p no:cacheprovider`.


In [ ]:
run_cmd([sys.executable, "-m", "pytest", "paper10_geojepa_mpc/tests", "-q", "-p", "no:cacheprovider"], cwd=REPO_DIR, log_path=LOG_DIR / "pytest.log")


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

RESULT_PREFIX = f"e0_frontier_random050_rank_seed2028_{n_states}x{candidate_actions}_h{label_horizon}_seed{label_seed}"
VALUE_HEAD_RUN = f"e0_frontier_random050_value_head_{n_states}x{candidate_actions}_h{label_horizon}_seed{label_seed}"

BASE_CHECKPOINT_PATH = REPO_DIR / "paper10_geojepa_mpc" / "experiments" / "checkpoints" / "e0_bishan_rank_seed2028" / "rank_seed2028.pt"
if not BASE_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Base checkpoint not found: {BASE_CHECKPOINT_PATH}")

LABEL_PATH = RUN_DIR / f"{RESULT_PREFIX}.npz"
LABEL_PARTIAL_PATH = RUN_DIR / f"{RESULT_PREFIX}.partial.npz"
VALUE_HEAD_DIR = RUN_DIR / "checkpoints" / VALUE_HEAD_RUN
VALUE_HEAD_DIR.mkdir(parents=True, exist_ok=True)
VALUE_HEAD_CHECKPOINT = VALUE_HEAD_DIR / f"value_head_seed{training_seed}.pt"
VALUE_HEAD_METRICS = RUN_DIR / f"{VALUE_HEAD_RUN}_metrics.json"

TRANSITION_PATH = DATA_ROOT_PATH / "tool2" / "transitions.npz"
TOOL2_PAIRWISE_PATH = DATA_ROOT_PATH / "tool2" / "pairwise.npz"

print(f"Label output: {LABEL_PATH}")
print(f"Partial label output: {LABEL_PARTIAL_PATH}")
print(f"Value-head checkpoint: {VALUE_HEAD_CHECKPOINT}")


## Generate 50x24/h5 value labels

Command preview: `python -X utf8 -m paper10_geojepa_mpc.experiments.value_label_generation --n-states 50 --candidate-actions 24 --label-horizon 5 --candidate-mode frontier_random --frontier-fraction 0.5 --partial-output ...partial.npz`.


In [ ]:
label_cmd = [
    sys.executable,
    "-X",
    "utf8",
    "-m",
    "paper10_geojepa_mpc.experiments.value_label_generation",
    "--checkpoint",
    str(BASE_CHECKPOINT_PATH),
    "--prepared-dir",
    str(DATA_ROOT_PATH),
    "--n-states",
    str(n_states),
    "--candidate-actions",
    str(candidate_actions),
    "--label-horizon",
    str(label_horizon),
    "--gamma",
    str(gamma),
    "--seed",
    str(label_seed),
    "--mask-mode",
    "executable",
    "--candidate-mode",
    candidate_mode,
    "--frontier-fraction",
    str(frontier_fraction),
    "--advance-policy",
    "random",
    "--continuation-policy",
    "random",
    "--score-batch-size",
    "512",
    "--device",
    device,
    "--partial-output",
    str(LABEL_PARTIAL_PATH),
    "--progress-every",
    str(progress_every),
    "--output",
    str(LABEL_PATH),
]
run_if_missing(LABEL_PATH, label_cmd, cwd=REPO_DIR, log_name="value_label_generation.log")


## Diagnostics and monitor gates

This cell writes top-3, top-4, and top-5 diagnostics and monitor files. Command previews include `--top-k 3`, `--top-k 4`, and `--top-k 5`. The automatic selector chooses the largest top-k whose monitor decision is `continue`; otherwise the run stops for inspection.


In [ ]:
monitor_results = {}
diagnostic_results = {}
for top_k in (3, 4, 5):
    diag_json = RUN_DIR / f"value_label_diagnostics_top{top_k}.json"
    diag_md = REPORT_DIR / f"value_label_diagnostics_top{top_k}.md"
    monitor_json = RUN_DIR / f"value_label_monitor_top{top_k}.json"
    monitor_md = REPORT_DIR / f"value_label_monitor_top{top_k}.md"

    run_if_missing(
        diag_json,
        [
            sys.executable,
            "-X",
            "utf8",
            "-m",
            "paper10_geojepa_mpc.experiments.value_label_diagnostics",
            "--input",
            str(LABEL_PATH),
            "--top-k",
            str(top_k),
            "--output-json",
            str(diag_json),
            "--output-md",
            str(diag_md),
        ],
        cwd=REPO_DIR,
        log_name=f"value_label_diagnostics_top{top_k}.log",
    )
    run_if_missing(
        monitor_json,
        [
            sys.executable,
            "-X",
            "utf8",
            "-m",
            "paper10_geojepa_mpc.experiments.value_label_monitor",
            "--input",
            str(LABEL_PATH),
            "--top-k",
            str(top_k),
            "--output-json",
            str(monitor_json),
            "--output-md",
            str(monitor_md),
        ],
        cwd=REPO_DIR,
        log_name=f"value_label_monitor_top{top_k}.log",
    )
    diagnostic_results[top_k] = json.loads(diag_json.read_text(encoding="utf-8"))
    monitor_results[top_k] = json.loads(monitor_json.read_text(encoding="utf-8"))

print(json.dumps({k: v["decision"] for k, v in monitor_results.items()}, indent=2, sort_keys=True))


In [ ]:
# Automatic top-k selection from monitor decisions.
passing_topks = [top_k for top_k, result in monitor_results.items() if result.get("decision") == "continue"]
if not passing_topks:
    raise RuntimeError("No top-k monitor decision was continue. Inspect monitor reports before training.")
selected_top_k = max(passing_topks)
SELECTED_TOPK_PATH = RUN_DIR / "selected_top_k.json"
SELECTED_TOPK_PATH.write_text(
    json.dumps(
        {
            "selected_top_k": selected_top_k,
            "passing_topks": passing_topks,
            "monitor_decisions": {str(k): v.get("decision") for k, v in monitor_results.items()},
        },
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)
print(f"selected_top_k={selected_top_k}")
print(f"Training preview: --device {device} --candidate-top-k {selected_top_k}")


## Train value head

Training uses CUDA when available. Command preview includes `--device {device}` and `--candidate-top-k {selected_top_k}`.


In [ ]:
train_cmd = [
    sys.executable,
    "-X",
    "utf8",
    "-m",
    "paper10_geojepa_mpc.experiments.run_e0_value_head_train",
    "--transition-path",
    str(TRANSITION_PATH),
    "--pairwise-path",
    str(LABEL_PATH),
    "--init-checkpoint",
    str(BASE_CHECKPOINT_PATH),
    "--checkpoint-path",
    str(VALUE_HEAD_CHECKPOINT),
    "--output",
    str(VALUE_HEAD_METRICS),
    "--epochs",
    str(training_epochs),
    "--batch-size",
    str(training_batch_size),
    "--transition-samples",
    str(transition_samples),
    "--pairwise-states",
    str(pairwise_states),
    "--pairwise-subsample",
    str(pairwise_subsample),
    "--n-pairs",
    str(n_pairs),
    "--candidate-top-k",
    str(selected_top_k),
    "--candidate-batch-states",
    "1",
    "--candidate-max-states",
    str(pairwise_states),
    "--checkpoint-metric",
    "auto",
    "--checkpoint-mode",
    "min",
    "--seed",
    str(training_seed),
    "--device",
    device,
]
run_if_missing(VALUE_HEAD_METRICS, train_cmd, cwd=REPO_DIR, log_name="value_head_train.log")
if not VALUE_HEAD_CHECKPOINT.exists():
    raise FileNotFoundError(f"Training metrics exist but checkpoint is missing: {VALUE_HEAD_CHECKPOINT}")


## 20-step blend gates

Runs two 20-step gates for `blend0.05` and `blend0.10`, using `--rollout-steps 20`, `--selector value_filter`, `--candidate-score-mode blend`, `--candidate-value-weight 0.05`, and `--candidate-value-weight 0.10`.


In [ ]:
gate_results = {}
for blend_weight in blend_gate_weights:
    blend_tag = f"blend{int(round(blend_weight * 100)):03d}"
    output_path = RUN_DIR / f"{VALUE_HEAD_RUN}_top{selected_top_k}_{blend_tag}_h{rollout_horizon}_k{rollout_top_k}_seed{rollout_seed}_{gate_rollout_steps}step.json"
    cmd = [
        sys.executable,
        "-X",
        "utf8",
        "-m",
        "paper10_geojepa_mpc.experiments.run_e0_env_rollout_smoke",
        "--checkpoint",
        str(VALUE_HEAD_CHECKPOINT),
        "--prepared-dir",
        str(DATA_ROOT_PATH),
        "--rollout-steps",
        str(gate_rollout_steps),
        "--horizon",
        str(rollout_horizon),
        "--top-k",
        str(rollout_top_k),
        "--seed",
        str(rollout_seed),
        "--device",
        device,
        "--mask-mode",
        "executable",
        "--selector",
        "value_filter",
        "--candidate-score-mode",
        "blend",
        "--candidate-value-weight",
        f"{blend_weight:.2f}",
        "--output",
        str(output_path),
    ]
    run_if_missing(output_path, cmd, cwd=REPO_DIR, log_name=f"rollout_gate_{blend_tag}.log")
    gate_results[blend_weight] = json.loads(output_path.read_text(encoding="utf-8"))

selected_blend_weight = max(
    gate_results,
    key=lambda weight: (float(gate_results[weight].get("total_reward", 0.0)), weight),
)
GATE_DECISION_PATH = RUN_DIR / "selected_blend_gate.json"
GATE_DECISION_PATH.write_text(
    json.dumps(
        {
            "selected_blend_weight": selected_blend_weight,
            "gate_rewards": {f"{weight:.2f}": result.get("total_reward") for weight, result in gate_results.items()},
        },
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)
print(json.dumps(json.loads(GATE_DECISION_PATH.read_text(encoding="utf-8")), indent=2, sort_keys=True))


## 100-step seed0 rollout

Runs the selected blend gate for seed 0 with `--rollout-steps 100` and `--seed 0`.


In [ ]:
selected_blend_tag = f"blend{int(round(selected_blend_weight * 100)):03d}"
ROLLOUT_100_SEED0 = RUN_DIR / f"{VALUE_HEAD_RUN}_top{selected_top_k}_{selected_blend_tag}_h{rollout_horizon}_k{rollout_top_k}_seed0_{final_rollout_steps}step.json"
ROLLOUT_100_SEED0_SUMMARY = RUN_DIR / f"{ROLLOUT_100_SEED0.stem}_summary.json"

rollout_seed0_cmd = [
    sys.executable,
    "-X",
    "utf8",
    "-m",
    "paper10_geojepa_mpc.experiments.run_e0_env_rollout_smoke",
    "--checkpoint",
    str(VALUE_HEAD_CHECKPOINT),
    "--prepared-dir",
    str(DATA_ROOT_PATH),
    "--rollout-steps",
    str(final_rollout_steps),
    "--horizon",
    str(rollout_horizon),
    "--top-k",
    str(rollout_top_k),
    "--seed",
    str(rollout_seed),
    "--device",
    device,
    "--mask-mode",
    "executable",
    "--selector",
    "value_filter",
    "--candidate-score-mode",
    "blend",
    "--candidate-value-weight",
    f"{selected_blend_weight:.2f}",
    "--output",
    str(ROLLOUT_100_SEED0),
]
run_if_missing(ROLLOUT_100_SEED0, rollout_seed0_cmd, cwd=REPO_DIR, log_name="rollout_100step_seed0.log")

run_if_missing(
    ROLLOUT_100_SEED0_SUMMARY,
    [
        sys.executable,
        "-X",
        "utf8",
        "-m",
        "paper10_geojepa_mpc.experiments.rollout_summary",
        str(ROLLOUT_100_SEED0),
        "--output",
        str(ROLLOUT_100_SEED0_SUMMARY),
    ],
    cwd=REPO_DIR,
    log_name="rollout_100step_seed0_summary.log",
)


## Optional 100-step seeds1-4 rollout

Set `run_optional_seeds = True` in the parameters cell to run the optional multiseed extension. Command preview: `--seeds 1-4`.


In [ ]:
ROLLOUT_100_SEEDS_1_4 = RUN_DIR / f"{VALUE_HEAD_RUN}_top{selected_top_k}_{selected_blend_tag}_h{rollout_horizon}_k{rollout_top_k}_seeds1-4_{final_rollout_steps}step.json"

if run_optional_seeds:
    optional_cmd = [
        sys.executable,
        "-X",
        "utf8",
        "-m",
        "paper10_geojepa_mpc.experiments.run_e0_env_rollout_smoke",
        "--checkpoint",
        str(VALUE_HEAD_CHECKPOINT),
        "--prepared-dir",
        str(DATA_ROOT_PATH),
        "--rollout-steps",
        str(final_rollout_steps),
        "--horizon",
        str(rollout_horizon),
        "--top-k",
        str(rollout_top_k),
        "--seeds",
        rollout_seeds_optional,
        "--device",
        device,
        "--mask-mode",
        "executable",
        "--selector",
        "value_filter",
        "--candidate-score-mode",
        "blend",
        "--candidate-value-weight",
        f"{selected_blend_weight:.2f}",
        "--output",
        str(ROLLOUT_100_SEEDS_1_4),
    ]
    run_if_missing(ROLLOUT_100_SEEDS_1_4, optional_cmd, cwd=REPO_DIR, log_name="rollout_100step_seeds1-4.log")
else:
    print("Skipping optional seeds1-4 rollout. Set run_optional_seeds = True to run it.")


## Summary report

This cell writes a compact JSON summary and a Markdown report to Drive.


In [ ]:
summary = {
    "run_name": RUN_NAME,
    "repo_url": REPO_URL,
    "repo_branch": REPO_BRANCH,
    "data_layout": DATA_LAYOUT,
    "parameters": {
        "n_states": n_states,
        "candidate_actions": candidate_actions,
        "label_horizon": label_horizon,
        "label_seed": label_seed,
        "training_seed": training_seed,
        "selected_top_k": selected_top_k,
        "selected_blend_weight": selected_blend_weight,
        "device": device,
    },
    "artifacts": {
        "label_path": str(LABEL_PATH),
        "label_partial_path": str(LABEL_PARTIAL_PATH),
        "value_head_checkpoint": str(VALUE_HEAD_CHECKPOINT),
        "value_head_metrics": str(VALUE_HEAD_METRICS),
        "rollout_seed0": str(ROLLOUT_100_SEED0),
        "rollout_seed0_summary": str(ROLLOUT_100_SEED0_SUMMARY),
        "rollout_seeds1_4": str(ROLLOUT_100_SEEDS_1_4) if ROLLOUT_100_SEEDS_1_4.exists() else None,
    },
    "monitor_decisions": {str(k): v.get("decision") for k, v in monitor_results.items()},
}

if VALUE_HEAD_METRICS.exists():
    summary["value_head_metrics_brief"] = json.loads(VALUE_HEAD_METRICS.read_text(encoding="utf-8"))
if ROLLOUT_100_SEED0_SUMMARY.exists():
    summary["rollout_seed0_summary_brief"] = json.loads(ROLLOUT_100_SEED0_SUMMARY.read_text(encoding="utf-8"))

SUMMARY_JSON = RUN_DIR / "frontier_random050_50x24_h5_summary.json"
SUMMARY_MD = REPORT_DIR / "frontier_random050_50x24_h5_report.md"
SUMMARY_JSON.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")

report_lines = [
    "# Paper10 frontier_random050 50x24/h5 Colab report",
    "",
    f"Run name: `{RUN_NAME}`",
    f"Device: `{device}`",
    f"Selected top-k: `{selected_top_k}`",
    f"Selected blend weight: `{selected_blend_weight:.2f}`",
    "",
    "## Artifacts",
    "",
]
for key, value in summary["artifacts"].items():
    report_lines.append(f"- `{key}`: `{value}`")
report_lines.extend(["", "## Monitor decisions", ""])
for key, value in summary["monitor_decisions"].items():
    report_lines.append(f"- top-{key}: `{value}`")
SUMMARY_MD.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

print(SUMMARY_JSON)
print(SUMMARY_MD)


## Package outputs

Creates a ZIP archive in Google Drive for download.


In [ ]:
PACKAGE_PATH = PACKAGE_DIR / f"{RUN_NAME}_outputs.zip"

include_paths = [
    RUN_DIR / "data_layout.json",
    LABEL_PATH,
    LABEL_PARTIAL_PATH,
    SELECTED_TOPK_PATH,
    GATE_DECISION_PATH,
    VALUE_HEAD_METRICS,
    VALUE_HEAD_CHECKPOINT,
    ROLLOUT_100_SEED0,
    ROLLOUT_100_SEED0_SUMMARY,
    SUMMARY_JSON,
    SUMMARY_MD,
]
if ROLLOUT_100_SEEDS_1_4.exists():
    include_paths.append(ROLLOUT_100_SEEDS_1_4)

for directory in (LOG_DIR, REPORT_DIR):
    if directory.exists():
        include_paths.extend(path for path in directory.rglob("*") if path.is_file())

with zipfile.ZipFile(PACKAGE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    seen = set()
    for path in include_paths:
        path = Path(path)
        if not path.exists() or path in seen:
            continue
        seen.add(path)
        zf.write(path, arcname=str(path.relative_to(RUN_DIR)))

print(f"Package written: {PACKAGE_PATH}")
